# ラズパイ再生テスト用のモデル書き出し（Colab）
1. 「ランタイム → ランタイムのタイプを変更 → T4 GPU」を選ぶ
2. 左のファイル欄に `emg_force_scripts.zip` をアップロード
3. 上から順にセルを実行（「ランタイム → すべてのセルを実行」でも可）
4. 最後にできる `export.zip` をダウンロードして、ラズパイに送る

In [ ]:
# 1. スクリプトを展開し、必要なライブラリを入れる
!unzip -o emg_force_scripts.zip
!pip install -q tables onnx onnxruntime
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
# 2. データをダウンロード（被験者 03・04・05、約 1.3GB。すでにあれば飛ばす）
!python put_force_dl.py 03 04 05

In [ ]:
# 3. 設定：被験者とチャンネル数
SUBJECT = '03'
BAND = 'all'            # 24ch。実機（PiEEG-16）に合わせるなら 'elbow_wrist'（16ch）
OUT = 'export' if BAND == 'all' else 'export16'

In [ ]:
# 4. 4 つのモデルを学習して書き出す（1 つ数分）
for args in ['--model transformer --size S', '--model lstm --size S', '--model esn --size M', '--model ridge']:
    print('=====', args)
    !python export_model.py {args} --subject {SUBJECT} --band {BAND} --out {OUT} 2>&1 | grep -v Warning

In [ ]:
# 5. まとめてダウンロード
!zip -qr {OUT}.zip {OUT} pi_replay.py
from google.colab import files
files.download(f'{OUT}.zip')